In [48]:
import pandas as pd

from pypdf import PdfReader
from pathlib import Path
import pdfplumber

In [49]:
data_folder = Path("../data/raw")

csv_files = list(data_folder.glob("*.csv"))
pdf_files = list(data_folder.glob("*.pdf"))

print("CSV files:")
for file in csv_files:
    print(file)

print("\nPDF files:")
for file in pdf_files:
    print(file)

CSV files:
..\data\raw\2022-COMMUNITY-PROJECTS-PETAUKE-CENTRAL.csv
..\data\raw\2025-NOT-APPROVED-COMMUNITY-PROJECTS-KAUMBWE2.csv
..\data\raw\2025-PROPOSED-COMMUNITY-PROJECTS-KAUMBWE2.csv

PDF files:
..\data\raw\11th-July-2025-Council-Minutes.pdf
..\data\raw\30th-April-2025- Council-Minutes.pdf
..\data\raw\Petauke-Town-Council-2025-OBB-Final-05.12.2024_Signed.pdf
..\data\raw\Petauke-Town-Council-2026-Budget-2026.pdf
..\data\raw\Petauke-Town-Council-Stratplan_2019-23.pdf
..\data\raw\Petauke.Lusangazi-Joint-IDP-Final.pdf


In [17]:
csv_data = {}

for file in csv_files:
    df = pd.read_csv(file)
    csv_data[file.name] = df

print(csv_data.keys())

dict_keys(['2022-COMMUNITY-PROJECTS-PETAUKE-CENTRAL.csv', '2025-NOT-APPROVED-COMMUNITY-PROJECTS-KAUMBWE2.csv', '2025-PROPOSED-COMMUNITY-PROJECTS-KAUMBWE2.csv'])


In [20]:
pdf_data = {}

for file in pdf_files:
    reader = PdfReader(file)

    text = ""

    for page in reader.pages:
        text += page.extract_text() or ""

    pdf_data[file.name] = text

print(pdf_data.keys())

dict_keys(['Petauke-Town-Council-2025-OBB-Final-05.12.2024_Signed.pdf', 'Petauke-Town-Council-2026-Budget-2026.pdf'])




INSPECTING "2026 BUDGET" AND "2025 OBB FINAL"

In [55]:
data_folder = Path("../data/raw") 
output_folder = Path("../data/processed") 
output_folder.mkdir(parents=True, exist_ok=True)

In [73]:
for pdf_file in pdf_files:

    if "2025-OBB" in pdf_file.name or "2026-Budget" in pdf_file.name:

        print(f"\nProcessing: {pdf_file.name}")

        extracted_tables = []

        with pdfplumber.open(pdf_file) as pdf:

            for page_num, page in enumerate(pdf.pages, start=1):

                tables = page.extract_tables()

                for table in tables:

                    if table:

                        df = pd.DataFrame(table)

                        # Remove empty rows and columns
                        df = df.fillna("")
                        df = df.map(
                            lambda cell: cell.strip()
                            if isinstance(cell, str)
                            else cell
                        )

                        df = df.loc[~(df == "").all(axis=1)]
                        df = df.loc[:, ~(df == "").all(axis=0)]

                        if not df.empty:
                            df["source_page"] = page_num
                            extracted_tables.append(df)

        # Save extracted tables
        if extracted_tables:

            final_df = pd.concat(
                extracted_tables,
                ignore_index=True
            )

            clean_name = (
                pdf_file.stem
                .lower()
                .replace("-", "_")
                .replace(".", "_")
            )

            output_file = (
                output_folder /
                f"db-unza26-csc4792-{clean_name}.csv"
            )

            final_df.to_csv(
                output_file,
                sep="|",
                index=False
            )

            print(f"Saved: {output_file}")
            print(f"Rows: {len(final_df)}")
            print(f"Columns: {len(final_df.columns)}")

        else:
            print("No tables found.")


Processing: Petauke-Town-Council-2025-OBB-Final-05.12.2024_Signed.pdf
Saved: ..\data\processed\db-unza26-csc4792-petauke_town_council_2025_obb_final_05_12_2024_signed.csv
Rows: 422
Columns: 9

Processing: Petauke-Town-Council-2026-Budget-2026.pdf
Saved: ..\data\processed\db-unza26-csc4792-petauke_town_council_2026_budget_2026.csv
Rows: 446
Columns: 9


In [ ]:
processed_files = list(output_folder.glob("*.csv"))

print("Processed CSV files:")

for file in processed_files:
    print(file)

In [ ]:
obb_dataset = pd.read_csv(
    "../data/processed/db-unza26-csc4792-petauke_town_council_2025_obb_final_05_12_2024_signed.csv",
    sep="|"
)

obb_dataset.head()

In [ ]:
budget_dataset = pd.read_csv(
    "../data/processed/db-unza26-csc4792-petauke_town_council_2026_budget_2026.csv",
    sep="|"
)

budget_dataset


In [68]:
obb_dataset.columns = obb_dataset.iloc[0]

obb_dataset = obb_dataset.iloc[1:].reset_index(drop=True)

In [ ]:
obb_dataset

## 4. PDF Extraction and Inspection

This section loads and inspects the two primary PDF documents used in this
project:

- **Petauke Town Council Strategic Plan 2019–2023**
- **Petauke–Lusangazi Joint Integrated Development Plan (IDP)**

Objectives:
- Extract raw text from each PDF.
- Identify the document's structure (sections, headings, tables).
- Examine content relevant to the dataset (projects, strategies, budgets).
- Note potential data quality issues (page headers/footers, formatting artifacts).

### 4.1 Petauke Town Council Strategic Plan 2019–2023

**Source:** `data/raw/Petauke-Town-Council-Stratplan_2019-23.pdf`

The Strategic Plan is loaded and its text extracted to inspect its
structure, identify sections and tables, and note data quality issues.

In [15]:
from pypdf import PdfReader

stratplan_path = data_folder / "Petauke-Town-Council-Stratplan_2019-23.pdf"
stratplan_reader = PdfReader(str(stratplan_path))

stratplan_text = ""
for page in stratplan_reader.pages:
    stratplan_text += (page.extract_text() or "") + "\n"

print(f"File        : {stratplan_path.name}")
print(f"Pages       : {len(stratplan_reader.pages)}")
print(f"Total chars : {len(stratplan_text):,}")

File        : Petauke-Town-Council-Stratplan_2019-23.pdf
Pages       : 72
Total chars : 124,985


#### 4.1.1 Preview

First 3000 characters of the extracted text, to confirm the document is a
text-based PDF (not a scanned image).

In [16]:
print(stratplan_text[:3000])

 STRATEGIC PLAN 2019-2023 
“A Clean, Healthy, Green Town that is connected and Inclusive of 
All in Local Economic Growth”
PETAUKE TOWN COUNCIL 

Table of Contents  
 
Table of Contents ...................................................................................................................................... ii 
Foreword .................................................................................................................................................. iii 
Preface .....................................................................................................................................................iv  
Acknowledgement ................................................................................................................................ .v 
Acronyms ................................................................................................................................................. vi 
List of Tables .............................

#### 4.1.2 Table detection

Scan all pages of the Strategic Plan and count how many tables are present.
This informs how much structured tabular data can be extracted.

In [17]:
import pdfplumber

with pdfplumber.open(stratplan_path) as pdf:
    total_tables = 0
    for page in pdf.pages:
        for t in page.extract_tables():
            if t:
                total_tables += 1
    print(f"Pages        : {len(pdf.pages)}")
    print(f"Total tables : {total_tables}")

Pages        : 72
Total tables : 179


### 4.2 Petauke–Lusangazi Joint Integrated Development Plan

**Source:** `data/raw/Petauke.Lusangazi-Joint-IDP-Final.pdf`

The Joint IDP is a 181-page document containing numerous project
implementation tables organised by chiefdom and constituency. It is the
richest source of structured project data for our dataset.

In [18]:
idp_path = data_folder / "Petauke.Lusangazi-Joint-IDP-Final.pdf"
idp_reader = PdfReader(str(idp_path))

idp_text = ""
for page in idp_reader.pages:
    idp_text += (page.extract_text() or "") + "\n"

print(f"File        : {idp_path.name}")
print(f"Pages       : {len(idp_reader.pages)}")
print(f"Total chars : {len(idp_text):,}")

File        : Petauke.Lusangazi-Joint-IDP-Final.pdf
Pages       : 181
Total chars : 274,372


#### 4.2.1 Preview

First 3000 characters of the extracted text.

In [19]:
print(idp_text[:3000])

1 
 
 
REPUBLIC OF ZAMBIA 
MINSITRY OF LOCAL GOVERNMENT 
 
JOINT INTEGRATED DEVELOPMENT PLAN  
 
FOR 
 
PETAUKE/LUSANGAZI DISTRICTS 
 
“Improved Social and Economic Welfare Through Well -Coordinated 
Climate Smart Investments and Sustainable Development By 2030.” 
 
PLANNING SURVEY AND KEY ISSUES REPORT, 
DEVELOPMENT FRAMEWORK AND IMPLEMENTATION 
PLAN 
 
Prepared By: IDP Technical Team  
 PETAUKE / LUSANGAZI  
TOWN COUNCILS 
 
 
2 
 
FOREWORD 
The development of the Joint Integrated 
Development Plan (IDP) for both Petauke and 
Lusangazi Districts gives the critical opportunities 
for the people of Petauke and Lusangazi to define 
their own destinies in terms of development.  
The participartory approach provided the platform for the people of the two districts to come 
up with pertinent issue s, develop projects and programmes with the Implementation plan 
which will make the two districts develop in pratical terms and at an accelerated rate. 
We congratulate the IDP technical team, t

#### 4.2.2 Table detection

Count the total number of tables across the IDP.

In [20]:
with pdfplumber.open(idp_path) as pdf:
    total_tables = 0
    for page in pdf.pages:
        for t in page.extract_tables():
            if t:
                total_tables += 1
    print(f"Pages        : {len(pdf.pages)}")
    print(f"Total tables : {total_tables}")

Pages        : 181
Total tables : 287


## 5. IDP Project Implementation Plan Extraction

The IDP contains a **project implementation plan** spanning pages 75–90.
It is organised into five chiefdom sections:

| Section  | Chiefdom      | Constituency                  |
|----------|---------------|-------------------------------|
| 3.3.2.1  | Kalindawalo   | Petauke Central Constituency  |
| 3.3.2.2  | Mumbi         | Petauke Central Constituency  |
| 3.3.2.3  | Mwanjawanthu  | Kaumbwe Constituency          |
| 3.3.2.4  | Nyamphande    | Msanzala Constituency         |
| 3.3.2.5  | Sandwe        | Msanzala Constituency         |

Each section's table lists identified projects with their sub-items. We
extract these into a flat dataset with one row per (project, sub-item).

### 5.1 Section header detection

Section headers follow the pattern:

    X.X.X.X <Chiefdom> Chiefdom (<Constituency> Constituency)

We scan the whole document and print every section header to confirm
which pages anchor each chiefdom's project table.

In [21]:
import re

section_re = re.compile(
    r"(\d+(?:\.\d+){1,4})\s+([A-Z][A-Za-z'’\- ]+?)\s+Chiefdom\s*\(([^)]+?)\)",
    re.IGNORECASE
)

# Pre-extract all page texts once — reused throughout the extraction.
with pdfplumber.open(idp_path) as pdf:
    all_page_texts = [p.extract_text() or "" for p in pdf.pages]

for page_num, text in enumerate(all_page_texts, start=1):
    m = section_re.search(text)
    if m:
        print(f"Page {page_num:3d}: {m.group(0).strip()}")

Page   7: 3.3.2.1 Kalindawalo Chiefdom (Petauke Central Constituency)
Page  74: 3.3.2.1 Kalindawalo Chiefdom (Petauke Central Constituency)
Page  76: 3.3.2.2 Mumbi Chiefdom (Petauke Central Constituency)
Page  80: 3.3.2.3 Mwanjawanthu Chiefdom (Kaumbwe Constituency)
Page  84: 3.3.2.4 Nyamphande Chiefdom (Msanzala Constituency)
Page  89: 3.3.2.5 Sandwe Chiefdom (Msanzala Constituency)
Page 128: 000.00
MUMBI CHIEFDOM (PETAUKE CENTRAL CONSTITUENCY)
Page 129: 000.00
Mwanjawanthu Chiefdom (Kaumbwe Constituency)


### 5.2 Parser

The parser converts each project table into a flat list of records, one
row per sub-item. It:

- Walks each candidate page.
- Tracks the nearest preceding section header.
- Detects project-start rows (S/N is a digit).
- Attaches subsequent rows to the current project as sub-items.
- Normalises cell values (removes bullets, newlines, extra whitespace).

In [22]:
def clean_cell(v):
    """Normalise a single cell value extracted from a PDF table."""
    if v is None:
        return ""
    s = str(v).replace("\uf0b7", "").replace("\n", " ")
    return " ".join(s.split()).strip()


def detect_section(text):
    """Return (section_num, chiefdom, constituency) or None."""
    m = section_re.search(text)
    if m:
        return (m.group(1).strip(), m.group(2).strip(), m.group(3).strip())
    return None


def parse_project_table(table, section, page_num):
    """Parse one IDP project table into a flat list of records."""
    if not table or len(table) < 2:
        return []

    sec_num, sec_chief, sec_const = section
    records = []
    current = None

    # Skip header rows
    data_start = 0
    for i, row in enumerate(table[:4]):
        first = clean_cell(row[0]).upper() if row else ""
        if "S/N" in first or "IDENTIFIED" in first:
            data_start = i + 1

    for row in table[data_start:]:
        if not any(row):
            continue

        sn = clean_cell(row[0]) if row else ""

        if sn.isdigit():
            # New project starts
            current = {
                "section_num":     sec_num,
                "chiefdom":        sec_chief,
                "constituency":    sec_const,
                "sn":              sn,
                "project_name":    clean_cell(row[1]) if len(row) > 1 else "",
                "village":         clean_cell(row[2]) if len(row) > 2 else "",
                "ward":            clean_cell(row[3]) if len(row) > 3 else "",
                "chiefdom_in_row": clean_cell(row[4]) if len(row) > 4 else "",
                "resources":       clean_cell(row[7]) if len(row) > 7 else "",
                "priority":        clean_cell(row[8]) if len(row) > 8 else "",
                "sub_item":        "",
                "source_page":     page_num,
            }
        else:
            # Continuation / sub-item row
            if current is None:
                continue
            for cell in row:
                text = clean_cell(cell)
                if text:
                    rec = dict(current)
                    rec["sub_item"] = text
                    records.append(rec)

    # Keep last project even if it had no sub-items
    if current and not any(
        r["sn"] == current["sn"] and r["source_page"] == page_num
        for r in records
    ):
        records.append(current)

    return records

### 5.3 Run extraction

We apply the parser to all pages that contain project tables, looking up
to 3 pages back for the nearest section header (since headers sometimes
appear on the page immediately before the table).

In [23]:
candidate_page_numbers = [75, 76, 77, 78, 80, 81, 82, 85, 86, 89, 90]
all_records = []

with pdfplumber.open(idp_path) as pdf:
    for page_num in candidate_page_numbers:
        page = pdf.pages[page_num - 1]

        # Find nearest preceding section header (up to 3 pages back)
        section = None
        for back in range(0, 4):
            idx = page_num - 1 - back
            if idx < 0:
                break
            section = detect_section(all_page_texts[idx])
            if section:
                break

        if section is None:
            section = ("", "", "")

        for table in page.extract_tables():
            if table and len(table) > 2:
                all_records.extend(parse_project_table(table, section, page_num))

idp_projects_df = pd.DataFrame(all_records)

print(f"Rows    : {len(idp_projects_df)}")
print(f"Columns : {len(idp_projects_df.columns)}")
print(f"\nSections captured:")
print(
    idp_projects_df[["section_num", "chiefdom", "constituency"]]
    .drop_duplicates()
    .to_string(index=False)
)

Rows    : 191
Columns : 12

Sections captured:
section_num     chiefdom                 constituency
    3.3.2.1  Kalindawalo Petauke Central Constituency
    3.3.2.2        Mumbi Petauke Central Constituency
    3.3.2.3 Mwanjawanthu         Kaumbwe Constituency
    3.3.2.4   Nyamphande        Msanzala Constituency
    3.3.2.5       Sandwe        Msanzala Constituency


### 5.4 Preview

Display the first 15 records to verify the parsed structure.

In [24]:
display(idp_projects_df.head(15))

,section_num,chiefdom,constituency,sn,project_name,village,ward,chiefdom_in_row,resources,priority,sub_item,source_page
0,3.3.2.1,Kalindawalo,Petauke Central Constituency,2,Construction/Rehabilitation of Dams (Irrigatio...,Selected villages,All wards,Kalindawalo,,2,Streams,75
1,3.3.2.1,Kalindawalo,Petauke Central Constituency,2,Construction/Rehabilitation of Dams (Irrigatio...,Selected villages,All wards,Kalindawalo,,2,Unskilled Labour,75
2,3.3.2.1,Kalindawalo,Petauke Central Constituency,2,Construction/Rehabilitation of Dams (Irrigatio...,Selected villages,All wards,Kalindawalo,,2,Crushed stones,75
3,3.3.2.1,Kalindawalo,Petauke Central Constituency,2,Construction/Rehabilitation of Dams (Irrigatio...,Selected villages,All wards,Kalindawalo,,2,River/Building Sand,75
4,3.3.2.1,Kalindawalo,Petauke Central Constituency,2,Construction/Rehabilitation of Dams (Irrigatio...,Selected villages,All wards,Kalindawalo,,2,defunct dams,75
5,3.3.2.1,Kalindawalo,Petauke Central Constituency,6,Drilling of solar powered of Boreholes,Villages with critical shortage of water,All Wards,Kalindawalo,,6,Unskilled Labour,75
6,3.3.2.1,Kalindawalo,Petauke Central Constituency,6,Drilling of solar powered of Boreholes,Villages with critical shortage of water,All Wards,Kalindawalo,,6,Crushed stones,75
7,3.3.2.1,Kalindawalo,Petauke Central Constituency,6,Drilling of solar powered of Boreholes,Villages with critical shortage of water,All Wards,Kalindawalo,,6,River/Building Sand,75
8,3.3.2.1,Kalindawalo,Petauke Central Constituency,7,Construction and Upgrading of Health Centers.,Selected villages,All Wards,Kalindawalo,,7,,75
9,3.3.2.2,Mumbi,Petauke Central Constituency,8,"Construction of Schools, Literacy Centres and ...",Selected villages,All Wards,Kalindawalo,,8,Unskilled Labour,76


### 5.5 Data inspection

Check for missing values and duplicates in the extracted dataset.

In [25]:
print("Shape:", idp_projects_df.shape)
print("\nMissing values per column:")
print(idp_projects_df.isnull().sum())
print("\nDuplicate rows:", idp_projects_df.duplicated().sum())

Shape: (191, 12)

Missing values per column:
section_num        0
chiefdom           0
constituency       0
sn                 0
project_name       0
village            0
ward               0
chiefdom_in_row    0
resources          0
priority           0
sub_item           0
source_page        0
dtype: int64

Duplicate rows: 1


### 5.6 Save to processed folder

The extracted dataset is saved as a pipe-separated CSV following the
assignment naming convention `db-unza26-csc4792-<description>.csv`.

In [26]:
from pathlib import Path

output_folder = Path("../data/processed")
output_folder.mkdir(parents=True, exist_ok=True)

output_file = output_folder / "db-unza26-csc4792-petauke_lusangazi_idp_projects.csv"

idp_projects_df.to_csv(
    output_file,
    sep="|",
    index=False,
    encoding="utf-8"
)

print(f"Saved  : {output_file}")
print(f"Rows   : {len(idp_projects_df)}")
print(f"Columns: {len(idp_projects_df.columns)}")

Saved  : ..\data\processed\db-unza26-csc4792-petauke_lusangazi_idp_projects.csv
Rows   : 191
Columns: 12


## 6. Strategic Plan 2019–2023 — Extraction and Inspection

The Strategic Plan 2019–2023 is a 72-page document outlining the Petauke
Town Council's development vision, strategies, and project priorities for
the period 2019–2023.

Unlike the IDP (which had one clearly structured project implementation
plan), the Strategic Plan contains a mix of narrative sections and data
tables scattered across different topics (population, employment, health,
education, infrastructure).

**Source:** `data/raw/Petauke-Town-Council-Stratplan_2019-23.pdf`

### 6.1 Full document structure

We begin by listing every table in the Strategic Plan with its page
number, dimensions, and a preview of its header row. This tells us which
tables contain real data (as opposed to page headers/footers).

In [27]:
import pdfplumber
import pandas as pd
import re
from pathlib import Path

data_folder = Path("../data/raw")
output_folder = Path("../data/processed")
output_folder.mkdir(parents=True, exist_ok=True)

stratplan_path = data_folder / "Petauke-Town-Council-Stratplan_2019-23.pdf"

def list_all_tables(pdf_path):
    """List every extracted table in a PDF with page, dimensions, and header preview."""
    rows = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            for t_idx, table in enumerate(page.extract_tables()):
                if not table:
                    continue
                header = table[0]
                header_str = " | ".join(
                    (str(c) or "")[:30] for c in header
                )
                rows.append({
                    "page":      page_num,
                    "table_idx": t_idx,
                    "rows":      len(table),
                    "cols":      len(header),
                    "header":    header_str,
                })
    return pd.DataFrame(rows)

stratplan_tables = list_all_tables(stratplan_path)
print(f"Total tables found: {len(stratplan_tables)}")

# Show tables with at least 3 rows and 2 cols (likely real data)
real_candidates = stratplan_tables[
    (stratplan_tables["rows"] >= 3) &
    (stratplan_tables["cols"] >= 2)
]
print(f"Tables with ≥3 rows and ≥2 cols: {len(real_candidates)}\n")
display(real_candidates.head(40))

Total tables found: 179
Tables with ≥3 rows and ≥2 cols: 30



,page,table_idx,rows,cols,header
0,1,0,5,3,None | | None
20,11,1,15,20,Table.1: Projected Percent urb | None | None |...
23,12,1,16,23,tcirtsiD\nekuateP\n5202-1102\nnoi | r aeY\nnoi...
26,13,1,11,7,| Labour Force | Persons In\nEmployment | Per...
31,15,1,11,18,S/N | | Type o\nlicense | Type o | f | 2016\n...
33,15,3,11,4,SN | Disease | All age\ncases | All age\nincid...
40,18,1,16,3,S/N | AREA/ LOCATION | No. OF CONNECTIONS
43,19,1,10,5,SN | Description | No of premises | No of Insp...
48,21,1,21,28,National 2016/2017 Vs 2017/201 | None | None |...
51,22,1,14,5,Number Of Livestock By Provinc | None | None |...


### 6.2 Filtering out page decoration

Many "tables" in the Strategic Plan are actually page headers and footers
containing the same text ("PETAUKE TOWN COUNCIL", "STRATEGIC PLAN
2019-2023", page numbers). We filter these out to surface only tables
that contain substantive data.

In [28]:
# Heuristics to detect and drop decoration tables
def is_decoration(row):
    header = str(row["header"]).lower()
    header_clean = header.replace("|", "").replace(" ", "").strip()

    # Headers that are literally just the council name
    if header_clean in ("petauketowncouncil", "petauke town council"):
        return True
    # Headers that contain page number markers
    if re.match(r"^\s*\|?\s*page\s+[ivx\d]+\s*\|", header):
        return True
    # Single-column tables — usually headers/footers
    if row["cols"] < 2:
        return True
    return False

stratplan_real = stratplan_tables[~stratplan_tables.apply(is_decoration, axis=1)]
stratplan_real = stratplan_real[stratplan_real["rows"] >= 3]

print(f"Real data tables: {len(stratplan_real)}\n")
display(stratplan_real.reset_index(drop=True))

Real data tables: 30



,page,table_idx,rows,cols,header
0,1,0,5,3,None | | None
1,11,1,15,20,Table.1: Projected Percent urb | None | None |...
2,12,1,16,23,tcirtsiD\nekuateP\n5202-1102\nnoi | r aeY\nnoi...
3,13,1,11,7,| Labour Force | Persons In\nEmployment | Per...
4,15,1,11,18,S/N | | Type o\nlicense | Type o | f | 2016\n...
5,15,3,11,4,SN | Disease | All age\ncases | All age\nincid...
6,18,1,16,3,S/N | AREA/ LOCATION | No. OF CONNECTIONS
7,19,1,10,5,SN | Description | No of premises | No of Insp...
8,21,1,21,28,National 2016/2017 Vs 2017/201 | None | None |...
9,22,1,14,5,Number Of Livestock By Provinc | None | None |...


### 6.3 Previewing candidate tables for extraction

We preview the raw row structure of three high-value tables from the
Strategic Plan (population projections, employment, and health
statistics) to plan extraction.

In [29]:
with pdfplumber.open(stratplan_path) as pdf:
    for page_num, table_idx in [(11, 1), (13, 1), (15, 3)]:
        page = pdf.pages[page_num - 1]
        tables = page.extract_tables()
        if table_idx >= len(tables):
            print(f"\nPage {page_num} — no table at index {table_idx}")
            continue
        table = tables[table_idx]
        print(f"\n{'='*70}")
        print(f"PAGE {page_num} — table {table_idx} ({len(table)} rows × {len(table[0])} cols)")
        print(f"{'='*70}")
        for row in table[:8]:
            print(row)


PAGE 11 — table 1 (15 rows × 20 cols)
['Table.1: Projected Percent urban Population By Province and Year of Projection(Medium Varian), Zambia 2011-\n2035', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]
['Province', None, None, 'Year of Projection', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]
[None, None, None, '2011', None, None, '2015', None, None, '2020', None, '2025', None, None, '2030', None, None, '2035', None, None]
['Central', None, None, '25.5', None, None, '25.4', None, None, '25.4', None, '25.3', None, None, '25.0', None, None, '24.5', None, None]
['Copperbelt', None, None, '82.1', None, None, '83.0', None, None, '84.1', None, '85.3', None, None, '86.2', None, None, '87.1', None, None]
['', 'Eastern', '', '', '11.9', '', '', '12.2', '', '', '12.7', '', '13.1', '', '', '13.1', '', '', '12.9', '']
['Luapula', None, None, '19.4', None, None, '21.0', None, None

### 6.4 Extracting selected Strategic Plan tables

We extract three high-value tables from the Strategic Plan:

1. **Population projections by province** (page 11)
2. **Employment statistics** (page 13)
3. **Health statistics** (page 15)

Each table has a different shape, so we write a small targeted parser for
each. All outputs are saved as pipe-separated CSVs following the
assignment naming convention.

In [30]:
# ---------- 1. Population projections (page 11) ----------

with pdfplumber.open(stratplan_path) as pdf:
    pop_table = pdf.pages[10].extract_tables()[1]  # 0-indexed page 11, table 1

def clean_cell(v):
    if v is None:
        return ""
    return " ".join(str(v).replace("\uf0b7", "").replace("\n", " ").split()).strip()

# The table has 3-column groups: [Province|None|None], [Year|None|None], [Percent|None|None]
# We'll skip None columns and build records
pop_records = []

# Row 0 is the title — skip
# Row 1 = headers, Row 2 = years, Row 3+ = data

# Get years from row 2 (index 2)
years_row = pop_table[2] if len(pop_table) > 2 else []
years = []
for i in range(3, len(years_row), 3):
    y = clean_cell(years_row[i]) if i < len(years_row) else ""
    years.append(y)

# Get province data starting from row 3
for row in pop_table[3:]:
    province = clean_cell(row[0]) or clean_cell(row[1]) if len(row) > 1 else ""
    if not province:
        continue
    # Extract percentages starting at index 3
    for idx, year in enumerate(years):
        col = 3 + idx * 3
        if col >= len(row):
            break
        pct = clean_cell(row[col])
        if pct and year:
            pop_records.append({
                "province":    province,
                "year":        year,
                "urban_pct":   pct,
                "source_page": 11,
            })

pop_df = pd.DataFrame(pop_records)
print(f"Population records: {len(pop_df)}")
display(pop_df.head(15))

Population records: 30


,province,year,urban_pct,source_page
0,Central,2011,25.5,11
1,Central,2015,25.4,11
2,Central,2020,25.4,11
3,Copperbelt,2011,82.1,11
4,Copperbelt,2015,83.0,11
5,Copperbelt,2020,84.1,11
6,Luapula,2011,19.4,11
7,Luapula,2015,21.0,11
8,Luapula,2020,23.1,11
9,Lusaka,2011,85.3,11


In [31]:
# ---------- 2. Employment statistics (page 13) ----------

with pdfplumber.open(stratplan_path) as pdf:
    emp_table = pdf.pages[12].extract_tables()[1]  # page 13, table 1

emp_records = []
perspective = ""

# Row 0 = column headers
# Rows 1+ = data, with markers like "International Perspective", "Local Perspective"
for row in emp_table:
    if not row:
        continue

    first = clean_cell(row[0])
    second = clean_cell(row[1]) if len(row) > 1 else ""

    # Detect perspective markers (in col 0 or col 1)
    marker = first or second
    if marker.lower() in ("international perspective", "local perspective"):
        perspective = marker
        continue

    # Data rows start with Total/Male/Female in col 0
    if first.lower() in ("total", "male", "female"):
        emp_records.append({
            "perspective":           perspective,
            "category":              first,
            "labour_force":          clean_cell(row[1]) if len(row) > 1 else "",
            "persons_in_employment": clean_cell(row[2]) if len(row) > 2 else "",
            "persons_in_unemployment": clean_cell(row[3]) if len(row) > 3 else "",
            "potential_labour_force":  clean_cell(row[4]) if len(row) > 4 else "",
            "source_page":           13,
        })

emp_df = pd.DataFrame(emp_records)
print(f"Employment records: {len(emp_df)}")
display(emp_df)

Employment records: 6


,perspective,category,labour_force,persons_in_employment,persons_in_unemployment,potential_labour_force,source_page
0,International Perspective,Total,"3,398,294","2,971,169","427,125",,13
1,International Perspective,Male,"2,041,306","1,797,957","243,349",,13
2,International Perspective,Female,"1,356,988","1,173,212","183,776",,13
3,Local Perspective,Total,"5,049,059","2,971,169","427,125","1,650,764",13
4,Local Perspective,Male,"2,759,098","1,797,957","243,349","717,792",13
5,Local Perspective,Female,"2,289,961","1,173,212","183,776","932,972",13


In [32]:
# ---------- 3. Health statistics (page 15) ----------

with pdfplumber.open(stratplan_path) as pdf:
    health_table = pdf.pages[14].extract_tables()[3]  # page 15, table 3

health_records = []
for row in health_table:
    if not row:
        continue
    sn = clean_cell(row[0])
    if not sn.isdigit():
        continue
    health_records.append({
        "sn":             sn,
        "disease":        clean_cell(row[1]) if len(row) > 1 else "",
        "cases":          clean_cell(row[2]) if len(row) > 2 else "",
        "incidence_rate": clean_cell(row[3]) if len(row) > 3 else "",
        "source_page":    15,
    })

health_df = pd.DataFrame(health_records)
print(f"Health records: {len(health_df)}")
display(health_df)

Health records: 10


,sn,disease,cases,incidence_rate,source_page
0,1,Respiratory infections,"51, 309",181,15
1,2,Malaria conrmed cases,"41, 419",173,15
2,3,Muscular skeleton & connective tissues,"10, 745",38,15
3,4,Digestive systems,"7, 046",25,15
4,5,Diarrhea non blood,"6, 152",22,15
5,6,Pyrexia of unknown origin (PUO),"3, 895",14,15
6,7,"Trauma, other injuries & wounds","2, 547",9,15
7,8,Skin diseases not infectious,"2, 473",9,15
8,9,dental carries,"1, 666",6,15
9,10,Throat diseases,"1, 509",5,15


### 6.5 Save Strategic Plan datasets to processed folder

Each extracted dataset is saved as a pipe-separated CSV following the
assignment naming convention `db-unza26-csc4792-<description>.csv`.

In [33]:
from pathlib import Path

output_folder = Path("../data/processed")
output_folder.mkdir(parents=True, exist_ok=True)

# --- 1. Population ---
pop_file = output_folder / "db-unza26-csc4792-petauke_stratplan_population.csv"
pop_df.to_csv(pop_file, sep="|", index=False, encoding="utf-8")
print(f"Saved: {pop_file.name}  ({len(pop_df)} rows)")

# --- 2. Employment ---
emp_file = output_folder / "db-unza26-csc4792-petauke_stratplan_employment.csv"
emp_df.to_csv(emp_file, sep="|", index=False, encoding="utf-8")
print(f"Saved: {emp_file.name}  ({len(emp_df)} rows)")

# --- 3. Health ---
health_file = output_folder / "db-unza26-csc4792-petauke_stratplan_health.csv"
health_df.to_csv(health_file, sep="|", index=False, encoding="utf-8")
print(f"Saved: {health_file.name}  ({len(health_df)} rows)")

Saved: db-unza26-csc4792-petauke_stratplan_population.csv  (30 rows)
Saved: db-unza26-csc4792-petauke_stratplan_employment.csv  (6 rows)
Saved: db-unza26-csc4792-petauke_stratplan_health.csv  (10 rows)


## 7. IDP Ward List Extraction

The Joint IDP contains a ward list on page 27 describing the wards that
make up each constituency. This supports the assignment's requirement
for ward development committee information.

**Source:** `data/raw/Petauke.Lusangazi-Joint-IDP-Final.pdf` — page 27

In [34]:
with pdfplumber.open(idp_path) as pdf:
    page = pdf.pages[26]   # page 27 (0-indexed 26)
    tables = page.extract_tables()

print(f"Tables on page 27: {len(tables)}\n")
for i, t in enumerate(tables):
    print(f"--- Table {i} ({len(t)} rows × {len(t[0]) if t else 0} cols) ---")
    for row in t[:20]:
        print(row)
    print()

Tables on page 27: 1

--- Table 0 (23 rows × 12 cols) ---
['S/N', None, None, 'Ward', 'Constituency', 'District', '', 'Current', '', '', 'Desired Status by', '']
[None, None, None, None, None, None, None, 'Status', None, None, 'the Community', None]
['1', None, None, 'Kaumbwe Kaumbwe Petauke 1 3 Rural Health\nCenters\n1 hospital', None, None, None, None, None, None, None, None]
['', '2', '', 'Ongolwe Petauke Central Petauke 2 4', None, None, None, None, None, None, None, None]
['', '3', '', 'Lusinde Kaumbwe Petauke 1 3', None, None, None, None, None, None, None, None]
['', '4', '', 'Mawanda Msanzala Lusangazi 4 8', None, None, None, None, None, None, None, None]
['5', '5', None, 'Mbala Petauke Central Petauke 3 4\n1general hospital', None, None, None, None, None, None, None, None]
['', '6', '', 'Mateyo Mzeka Petauke Central Petauke 3 3', None, None, None, None, None, None, None, None]
['', '7', '', 'Chisangu Msanzala Lusangazi 1 3', None, None, None, None, None, None, None, None]
['', 

### 7.1 Full preview of the ward list table

The ward list table on page 27 has 23 rows × 12 columns. Most cells are
None (padding), and the Ward column appears to contain concatenated
fields. We preview the full table to identify the parsing rules.

In [35]:
with pdfplumber.open(idp_path) as pdf:
    page = pdf.pages[26]
    table = page.extract_tables()[0]

print(f"Total rows: {len(table)}\n")
for i, row in enumerate(table):
    # Show only non-None cells with their column index
    nonempty = [(j, str(c).replace("\n", "\\n")[:70]) for j, c in enumerate(row) if c not in (None, "", " ")]
    print(f"Row {i:2d}: {nonempty}")

Total rows: 23

Row  0: [(0, 'S/N'), (3, 'Ward'), (4, 'Constituency'), (5, 'District'), (7, 'Current'), (10, 'Desired Status by')]
Row  1: [(7, 'Status'), (10, 'the Community')]
Row  2: [(0, '1'), (3, 'Kaumbwe Kaumbwe Petauke 1 3 Rural Health\\nCenters\\n1 hospital')]
Row  3: [(1, '2'), (3, 'Ongolwe Petauke Central Petauke 2 4')]
Row  4: [(1, '3'), (3, 'Lusinde Kaumbwe Petauke 1 3')]
Row  5: [(1, '4'), (3, 'Mawanda Msanzala Lusangazi 4 8')]
Row  6: [(0, '5'), (1, '5'), (3, 'Mbala Petauke Central Petauke 3 4\\n1general hospital')]
Row  7: [(1, '6'), (3, 'Mateyo Mzeka Petauke Central Petauke 3 3')]
Row  8: [(1, '7'), (3, 'Chisangu Msanzala Lusangazi 1 3')]
Row  9: [(1, '8'), (3, 'Msumbazi Petauke Central Petauke 1 6')]
Row 10: [(1, '9'), (3, 'Singozi Petauke Central Petauke 2 3')]
Row 11: [(1, '10'), (3, 'Manyane Kaumbwe Petauke 4 4')]
Row 12: [(1, '11'), (3, 'Ukwimi Msanzala Lusangazi 6 8')]
Row 13: [(1, '12'), (3, 'Chilimanyama Petauke Central Petauke 2 2')]
Row 14: [(1, '13'), (3, 'Ko

### 7.3 Parser for ward list

The ward column concatenates four fields:

    <Ward Name> <Constituency> <District> <N1> <N2> [<facility info>]

We use a regex to split them. The two trailing numbers are likely the
counts of the ward's sub-committees or the two figures used in the
"Current Status" and "Desired Status" columns of the source table.

Summary rows (e.g. "Total 46 76") are skipped.

In [36]:
import re

ward_re = re.compile(
    r"^(?P<ward>.+?)\s+"
    r"(?P<constituency>Kaumbwe|Petauke Central|Msanzala|Mumbi|Nyamphande|Sandwe)\s+"
    r"(?P<district>Petauke|Lusangazi)\s+"
    r"(?P<n1>\d+)\s+(?P<n2>\d+)"
    r"(?:\s*(?P<facility>.+))?$",
    re.IGNORECASE
)

ward_records = []

for i, row in enumerate(table):
    # Skip header rows
    if i < 2:
        continue

    # Get S/N — sometimes in col 0, sometimes col 1
    sn = ""
    if len(row) > 0 and row[0] and str(row[0]).strip():
        sn = str(row[0]).strip()
    elif len(row) > 1 and row[1] and str(row[1]).strip():
        sn = str(row[1]).strip()

    # Get ward string from column 3
    ward_raw = row[3] if len(row) > 3 and row[3] else ""
    if not ward_raw:
        continue

    ward_text = " ".join(str(ward_raw).replace("\n", " ").split())

    # Skip summary rows (e.g. "Total 46 76")
    if ward_text.lower().startswith("total"):
        continue

    m = ward_re.match(ward_text)
    if not m:
        # Fallback — store raw so nothing is lost
        ward_records.append({
            "sn":           sn,
            "ward":         ward_text,
            "constituency": "",
            "district":     "",
            "n1":           "",
            "n2":           "",
            "facility":     "",
            "source_page":  27,
        })
        continue

    ward_records.append({
        "sn":           sn,
        "ward":         m.group("ward").strip(),
        "constituency": m.group("constituency").strip(),
        "district":     m.group("district").strip(),
        "n1":           m.group("n1"),
        "n2":           m.group("n2"),
        "facility":     (m.group("facility") or "").strip(),
        "source_page":  27,
    })

ward_df = pd.DataFrame(ward_records)
print(f"Ward records: {len(ward_df)}")
display(ward_df)

Ward records: 20


,sn,ward,constituency,district,n1,n2,facility,source_page
0,1,Kaumbwe,Kaumbwe,Petauke,1,3,Rural Health Centers 1 hospital,27
1,2,Ongolwe,Petauke Central,Petauke,2,4,,27
2,3,Lusinde,Kaumbwe,Petauke,1,3,,27
3,4,Mawanda,Msanzala,Lusangazi,4,8,,27
4,5,Mbala,Petauke Central,Petauke,3,4,1general hospital,27
5,6,Mateyo Mzeka,Petauke Central,Petauke,3,3,,27
6,7,Chisangu,Msanzala,Lusangazi,1,3,,27
7,8,Msumbazi,Petauke Central,Petauke,1,6,,27
8,9,Singozi,Petauke Central,Petauke,2,3,,27
9,10,Manyane,Kaumbwe,Petauke,4,4,,27


### 7.4 Saving ward list to processed folder

The parsed ward list is saved as a pipe-separated CSV.

In [37]:
ward_file = output_folder / "db-unza26-csc4792-petauke_lusangazi_idp_wards.csv"
ward_df.to_csv(ward_file, sep="|", index=False, encoding="utf-8")

print(f"Saved: {ward_file.name}")
print(f"Rows : {len(ward_df)}")

# Verify
verify = pd.read_csv(ward_file, sep="|", dtype=str, keep_default_na=False)
print(f"Verify shape: {verify.shape}")
display(verify.head(5))

Saved: db-unza26-csc4792-petauke_lusangazi_idp_wards.csv
Rows : 20
Verify shape: (20, 8)


,sn,ward,constituency,district,n1,n2,facility,source_page
0,1,Kaumbwe,Kaumbwe,Petauke,1,3,Rural Health Centers 1 hospital,27
1,2,Ongolwe,Petauke Central,Petauke,2,4,,27
2,3,Lusinde,Kaumbwe,Petauke,1,3,,27
3,4,Mawanda,Msanzala,Lusangazi,4,8,,27
4,5,Mbala,Petauke Central,Petauke,3,4,1general hospital,27


## 6. IDP Location List Extraction

Pages 42–43 of the Joint IDP contain a list of named locations with
categories, areas, and perimeters. This provides a spatial reference for
project sites in the dataset.

**Source:** `data/raw/Petauke.Lusangazi-Joint-IDP-Final.pdf` — pages 42–43

In [38]:
with pdfplumber.open(idp_path) as pdf:
    for page_num in [42, 43]:
        page = pdf.pages[page_num - 1]
        tables = page.extract_tables()
        print(f"\n{'='*70}\nPAGE {page_num} — {len(tables)} table(s)\n{'='*70}")
        for i, t in enumerate(tables):
            print(f"\n--- Table {i} ({len(t)} rows × {len(t[0]) if t else 0} cols) ---")
            for row in t[:10]:
                print(row)


PAGE 42 — 1 table(s)

--- Table 0 (30 rows × 7 cols) ---
['S/', 'LOCATION', 'NAME', 'CATEG', 'AREA', 'PERIM', 'STATUS (HA']
['N', '', 'OF', 'ORY', '(HA)', 'ETER', 'encroached)']
['', None, '', '', '', '', '']
['1', 'Petauke', 'FOREST\nMupya', 'LF', '308', '(KM)\n7.3', 'About 261.8 HA has been']
[None, None, None, None, None, None, 'cleared for agriculture']
['', '', 'West', '', '', '', None]
[None, None, None, None, None, None, 'fields and settlements']
[None, None, '', None, None, None, None]
['2', 'Petauke', 'Kapungwe', 'LF', '1348', '17.4', 'About 808.8 HA']
['', '', 'West', '', '', '', 'encroached for agriculture']

PAGE 43 — 1 table(s)

--- Table 0 (11 rows × 7 cols) ---
['S/', 'LOCATION', 'NAME', 'CATEG', 'AREA', 'PERIM', 'STATUS (HA']
['N', '', 'OF', 'ORY', '(HA)', 'ETER', 'encroached)']
['', None, '', '', '', '', '']
['10', 'Petauke', 'FOREST\nMvuvye', 'NF', '35,937', '(KM)\n132', 'About 14,375.8 HA']
['', '', 'East', '', '', '', 'encroached in some parts']
[None, None, '', No

### 6.1 Parser for the location list

The location table on pages 42–43 uses multi-row records: a numeric S/N in
column 0 starts a new record; subsequent rows with an empty S/N contribute
additional text to the current record (locations like "Petauke West" and
multi-line status descriptions are split across rows).

Summary rows (e.g. "PETAUKE/LUSANGAZI TOTAL") are skipped.

In [39]:
import pandas as pd

def clean_cell(v):
    if v is None:
        return ""
    return " ".join(str(v).replace("\n", " ").split()).strip()


def parse_location_table(table, page_num):
    """Parse a location table from the Joint IDP into flat records."""
    records = []
    current = None

    for row in table:
        if not row:
            continue

        sn_raw = clean_cell(row[0])

        # Skip header rows (contain "S/" or "N" or "LOCATION")
        if sn_raw.upper() in ("S/", "N") or "LOCATION" in sn_raw.upper():
            continue

        # Skip total rows
        if "TOTAL" in sn_raw.upper():
            continue

        # Skip fully empty rows
        if not any(clean_cell(c) for c in row):
            continue

        # New record starts with a digit in col 0
        if sn_raw.isdigit():
            if current is not None:
                records.append(current)
            current = {
                "sn":           sn_raw,
                "location":     clean_cell(row[1]) if len(row) > 1 else "",
                "name":         clean_cell(row[2]) if len(row) > 2 else "",
                "category":     clean_cell(row[3]) if len(row) > 3 else "",
                "area_ha":      clean_cell(row[4]) if len(row) > 4 else "",
                "perimeter_km": clean_cell(row[5]) if len(row) > 5 else "",
                "status":       clean_cell(row[6]) if len(row) > 6 else "",
                "source_page":  page_num,
            }
        else:
            # Continuation row — append to current record
            if current is None:
                continue

            # If col 1 has text (like "West"), append to location
            if len(row) > 1 and clean_cell(row[1]):
                current["location"] = (
                    current["location"] + " " + clean_cell(row[1])
                ).strip()

            # If col 2 has text (like "West"), append to name
            if len(row) > 2 and clean_cell(row[2]):
                current["name"] = (
                    current["name"] + " " + clean_cell(row[2])
                ).strip()

            # If col 6 has text, append to status
            if len(row) > 6 and clean_cell(row[6]):
                current["status"] = (
                    current["status"] + " " + clean_cell(row[6])
                ).strip()

    if current is not None:
        records.append(current)

    return records


# Run it on both pages
all_locations = []
with pdfplumber.open(idp_path) as pdf:
    for page_num in [42, 43]:
        page = pdf.pages[page_num - 1]
        for table in page.extract_tables():
            if table:
                all_locations.extend(parse_location_table(table, page_num))

locations_df = pd.DataFrame(all_locations)

print(f"Rows    : {len(locations_df)}")
print(f"Columns : {list(locations_df.columns)}")
display(locations_df)

Rows    : 10
Columns : ['sn', 'location', 'name', 'category', 'area_ha', 'perimeter_km', 'status', 'source_page']


,sn,location,name,category,area_ha,perimeter_km,status,source_page
0,1,Petauke,FOREST Mupya West,LF,308,(KM) 7.3,About 261.8 HA has been cleared for agricultur...,42
1,2,Petauke,Kapungwe West,LF,1348,17.4,About 808.8 HA encroached for agriculture fields,42
2,3,Petauke,Luwenga,LF,1303,15.6,About 390.9 HA encroached d for agriculture,42
3,4,Petauke,Msumbazi,LF,"2,141",19.6,"fAieblodust 1,605.75 HA encroached for mostly",42
4,5,Petauke,Nsangwa North,LF,809,18.1,aAgbroicuutl 3tu6r4e. 0fi5e lHdsA a nd a few s...,42
5,6,Petauke,Nsangwa South,LF,"1,959",22.2,sAebttoleumt 5e8n7t .7 HA encroached with,42
6,7,Petauke,Mpamadzi,LF,791,13.2,sAebttoleumt 2e7n6ts. 8a5n dH fAie lds encroac...,42
7,8,Lusangazi,Sasare,LF,"2,600",26,"fAieblodust 1,300 HA encroached with fields, v...",42
8,9,Petauke,Minga,NF,"6,653",33.4,"sAtabtoiount 3,659.15 HA encroached with agric...",42
9,10,Petauke,FOREST Mvuvye East,NF,"35,937",(KM) 132,"About 14,375.8 HA encroached in some parts wit...",43


### 6.2 Save location list to processed folder

**Known data quality issue:** Some status strings extracted from pages 42–43
show interleaved characters (e.g. `fAieblodust` instead of `About fields`).
This is a pdfplumber text-layer artifact caused by overlapping PDF content
streams. These will be corrected during the Data Cleaning stage.

In [40]:
locations_file = output_folder / "db-unza26-csc4792-petauke_lusangazi_idp_locations.csv"
locations_df.to_csv(locations_file, sep="|", index=False, encoding="utf-8")

print(f"Saved: {locations_file.name}")
print(f"Rows : {len(locations_df)}")

Saved: db-unza26-csc4792-petauke_lusangazi_idp_locations.csv
Rows : 10


## 7. IDP Development Objectives Extraction

Page 71 of the Joint IDP contains a table of **development objectives**
with their corresponding developmental strategies. This is the strategic
framework that the IDP's projects are aligned to.

**Source:** `data/raw/Petauke.Lusangazi-Joint-IDP-Final.pdf` — page 71

In [41]:
with pdfplumber.open(idp_path) as pdf:
    page = pdf.pages[70]    # page 71 (0-indexed 70)
    tables = page.extract_tables()

print(f"Tables on page 71: {len(tables)}\n")
for i, t in enumerate(tables):
    print(f"{'='*70}\nTable {i} — {len(t)} rows × {len(t[0]) if t else 0} cols\n{'='*70}")
    for row in t[:15]:
        print(row)
    print()

Tables on page 71: 3

Table 0 — 18 rows × 5 cols
['S/N', 'Development Objectives', 'Developmental Strategies', None, None]
['1', 'To increase and diversify agriculture, livestock and\nfisheries production and productivity through proven\ntechnological innovations.', '', '• Diversification of agricultural, livestock and fisheries production', '']
[None, None, None, 'and utilization of modern technology.', None]
[None, None, None, '• Creation of crop demo plots and crop field schools.', None]
[None, None, None, '• Introduce intensive and commercial farming using smart', None]
[None, None, None, 'investments.', None]
[None, None, None, '• Introduce sustainable and environmentally sound agricultural', None]
[None, None, None, 'practices.', None]
[None, None, None, '• Strengthening emergency preparedness through timely early', None]
[None, None, None, 'warning and efficient crop focusing and maintenance of strategic', None]
[None, None, None, 'food reserves', None]
[None, None, None, '• Con

### 7.1 Full preview of the development objectives table

The table has 18 rows. We display all rows to confirm the pattern before
writing the parser.

In [42]:
with pdfplumber.open(idp_path) as pdf:
    table = pdf.pages[70].extract_tables()[0]

print(f"Total rows: {len(table)}\n")
for i, row in enumerate(table):
    nonempty = [(j, str(c).replace("\n", "\\n")[:75]) for j, c in enumerate(row) if c not in (None, "", " ")]
    print(f"Row {i:2d}: {nonempty}")

Total rows: 18

Row  0: [(0, 'S/N'), (1, 'Development Objectives'), (2, 'Developmental Strategies')]
Row  1: [(0, '1'), (1, 'To increase and diversify agriculture, livestock and\\nfisheries production '), (3, '• Diversification of agricultural, livestock and fisheries production')]
Row  2: [(3, 'and utilization of modern technology.')]
Row  3: [(3, '• Creation of crop demo plots and crop field schools.')]
Row  4: [(3, '• Introduce intensive and commercial farming using smart')]
Row  5: [(3, 'investments.')]
Row  6: [(3, '• Introduce sustainable and environmentally sound agricultural')]
Row  7: [(3, 'practices.')]
Row  8: [(3, '• Strengthening emergency preparedness through timely early')]
Row  9: [(3, 'warning and efficient crop focusing and maintenance of strategic')]
Row 10: [(3, 'food reserves')]
Row 11: [(3, '• Conduct continuous sensitization meetings to farmers on good')]
Row 12: [(3, '/best agriculture best practices.')]
Row 13: [(0, '2'), (1, 'To promote the conservation of nat

### 7.2 Parser for development objectives
Objective 2's strategy appears in column 2 rather than column 3, due to
a shifted cell in the source PDF. We extend the parser to check both
columns for strategy content.

In [44]:
def parse_objectives_table(table, page_num):
    """Parse the IDP development objectives table into (objective, strategy) rows."""
    records = []
    current_obj_num  = ""
    current_obj_text = ""
    current_strategy = ""

    def flush_strategy():
        if current_strategy:
            records.append({
                "sn":          current_obj_num,
                "objective":   current_obj_text.strip(),
                "strategy":    current_strategy.strip(),
                "source_page": page_num,
            })

    for row in table:
        if not row:
            continue

        sn   = clean_cell(row[0])
        col1 = clean_cell(row[1]) if len(row) > 1 else ""
        col2 = clean_cell(row[2]) if len(row) > 2 else ""
        col3 = clean_cell(row[3]) if len(row) > 3 else ""

        # Skip header
        if sn.upper() == "S/N" or "Development Objectives" in col1:
            continue

        # New objective
        if sn.isdigit():
            flush_strategy()
            current_strategy = ""
            current_obj_num  = sn
            current_obj_text = col1
            # Strategy might be in col 2 or col 3 on the same row
            if col2:
                current_strategy = col2
            if col3:
                current_strategy = (current_strategy + " " + col3).strip()
            continue

        # Continuation rows
        if col1:
            current_obj_text = (current_obj_text + " " + col1).strip()
        if col2:
            current_strategy = (current_strategy + " " + col2).strip()
        if col3:
            if col3.startswith("•") or col3.startswith("\uf0b7"):
                flush_strategy()
                current_strategy = col3.lstrip("•\uf0b7").strip()
            else:
                current_strategy = (current_strategy + " " + col3).strip()

    flush_strategy()
    return records


with pdfplumber.open(idp_path) as pdf:
    obj_table = pdf.pages[70].extract_tables()[0]

objectives_df = pd.DataFrame(parse_objectives_table(obj_table, page_num=71))
print(f"Rows: {len(objectives_df)}")
display(objectives_df)

Rows: 9


,sn,objective,strategy,source_page
0,1,"To increase and diversify agriculture, livesto...","• Diversification of agricultural, livestock a...",71
1,1,"To increase and diversify agriculture, livesto...",Creation of crop demo plots and crop field sch...,71
2,1,"To increase and diversify agriculture, livesto...",Introduce intensive and commercial farming usi...,71
3,1,"To increase and diversify agriculture, livesto...",Introduce sustainable and environmentally soun...,71
4,1,"To increase and diversify agriculture, livesto...",Strengthening emergency preparedness through t...,71
5,1,"To increase and diversify agriculture, livesto...",Conduct continuous sensitization meetings to f...,71
6,2,To promote the conservation of natural resources,• Creation of eco-friendly forests/fruit plant...,71
7,3,To develop infrastructure in the key sectors o..., Develop infrastructure in the key sectors of...,71
8,3,To develop infrastructure in the key sectors o...,Promotion of public private partnerships in th...,71


### 7.3 Saving development objectives to processed folder

In [45]:
objectives_file = output_folder / "db-unza26-csc4792-petauke_lusangazi_idp_objectives.csv"
objectives_df.to_csv(objectives_file, sep="|", index=False, encoding="utf-8")

print(f"Saved: {objectives_file.name}")
print(f"Rows : {len(objectives_df)}")

Saved: db-unza26-csc4792-petauke_lusangazi_idp_objectives.csv
Rows : 9


## 8. Data Cleaning

Each extracted dataset is loaded, inspected, and cleaned. Cleaned files
are saved to a separate `data/cleaned/` folder to preserve traceability:

- `data/raw/`       — original source files
- `data/processed/` — extracted but uncleaned DataFrames
- `data/cleaned/`   — cleaned, Kaggle-ready CSVs

### 8.1 Setup — Cleaned folder

In [46]:
from pathlib import Path

processed = Path("../data/processed")
cleaned   = Path("../data/cleaned")
cleaned.mkdir(parents=True, exist_ok=True)

print(f"Processed folder: {processed.resolve()}")
print(f"Cleaned folder  : {cleaned.resolve()}")
print(f"Cleaned folder exists: {cleaned.exists()}")

Processed folder: C:\Users\david\project_team_31\data\processed
Cleaned folder  : C:\Users\david\project_team_31\data\cleaned
Cleaned folder exists: True


### 8.2 Cleaning — IDP Projects

In [47]:
proj_file = processed / "db-unza26-csc4792-petauke_lusangazi_idp_projects.csv"
df = pd.read_csv(proj_file, sep="|", dtype=str, keep_default_na=False)

print("Shape:", df.shape)
print("Columns:", list(df.columns))
print("\nEmpty values per column:")
print((df == "").sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nSample:")
display(df.head(5))

Shape: (191, 12)
Columns: ['section_num', 'chiefdom', 'constituency', 'sn', 'project_name', 'village', 'ward', 'chiefdom_in_row', 'resources', 'priority', 'sub_item', 'source_page']

Empty values per column:
section_num          0
chiefdom             0
constituency         0
sn                   0
project_name         2
village             20
ward                62
chiefdom_in_row     73
resources          138
priority            48
sub_item             8
source_page          0
dtype: int64

Duplicate rows: 1

Sample:


,section_num,chiefdom,constituency,sn,project_name,village,ward,chiefdom_in_row,resources,priority,sub_item,source_page
0,3.3.2.1,Kalindawalo,Petauke Central Constituency,2,Construction/Rehabilitation of Dams (Irrigatio...,Selected villages,All wards,Kalindawalo,,2,Streams,75
1,3.3.2.1,Kalindawalo,Petauke Central Constituency,2,Construction/Rehabilitation of Dams (Irrigatio...,Selected villages,All wards,Kalindawalo,,2,Unskilled Labour,75
2,3.3.2.1,Kalindawalo,Petauke Central Constituency,2,Construction/Rehabilitation of Dams (Irrigatio...,Selected villages,All wards,Kalindawalo,,2,Crushed stones,75
3,3.3.2.1,Kalindawalo,Petauke Central Constituency,2,Construction/Rehabilitation of Dams (Irrigatio...,Selected villages,All wards,Kalindawalo,,2,River/Building Sand,75
4,3.3.2.1,Kalindawalo,Petauke Central Constituency,2,Construction/Rehabilitation of Dams (Irrigatio...,Selected villages,All wards,Kalindawalo,,2,defunct dams,75


### 8.2 Cleaning — IDP Projects

Issues identified:
- One exact duplicate row (drop).
- 8 rows with empty `sub_item` (projects with no sub-items — keep, this is
  meaningful).
- Leading/trailing whitespace in string columns (strip).
- `source_page` should be integer.

In [48]:
# Diagnose the duplicate
dup_mask = df.duplicated(keep=False)
print("Duplicate rows (all copies shown):")
display(df[dup_mask])

# Check whether the first 5 rows differ in sub_item
print("\nFirst 5 rows — sub_item values:")
print(df.loc[0:4, ["sn", "project_name", "sub_item"]].to_string())

Duplicate rows (all copies shown):


,section_num,chiefdom,constituency,sn,project_name,village,ward,chiefdom_in_row,resources,priority,sub_item,source_page
146,3.3.2.4,Nyamphande,Msanzala Constituency,8,Construction and Upgrading of Schools and housing,"Misolo, Nsamba, Nsenya, Ray, Mkonda, Mwambula,...",A ll W ar d s,A,N,,d,86
147,3.3.2.4,Nyamphande,Msanzala Constituency,8,Construction and Upgrading of Schools and housing,"Misolo, Nsamba, Nsenya, Ray, Mkonda, Mwambula,...",A ll W ar d s,A,N,,d,86



First 5 rows — sub_item values:
  sn                                               project_name             sub_item
0  2  Construction/Rehabilitation of Dams (Irrigation Schemes).              Streams
1  2  Construction/Rehabilitation of Dams (Irrigation Schemes).     Unskilled Labour
2  2  Construction/Rehabilitation of Dams (Irrigation Schemes).       Crushed stones
3  2  Construction/Rehabilitation of Dams (Irrigation Schemes).  River/Building Sand
4  2  Construction/Rehabilitation of Dams (Irrigation Schemes).         defunct dams


### 8.2 Cleaning — IDP Projects

Issues identified in inspection:

- **1 exact duplicate row** — drop.
- **Newlines inside cells** (`village` contains `\n`) — replace with `, `.
- **Garbled whitespace** in `ward` (e.g. `"A ll\nW ar\nd s"` instead of
  `"All Wards"`) — collapse whitespace and remove interleaved spaces.
- **Whitespace** — strip across all string columns.
- **`source_page`** — convert to integer.

The garbled `ward` strings are the same pdfplumber text-layer artifact
seen in `idp_locations.csv`. We apply a shared cleaner.

In [49]:
import re
import pandas as pd
from pathlib import Path

def clean_cell(v):
    """Normalise a single cell value."""
    if v is None:
        return ""
    s = str(v)
    s = s.replace("\uf0b7", "")            # bullet
    s = s.replace("\n", ", ")               # newlines in cells -> comma
    s = " ".join(s.split())                 # collapse whitespace
    return s.strip(" ,")

def fix_interleaved_ward(s):
    """
    Fix interleaved whitespace like 'A ll W ar d s' -> 'All Wards'.
    Heuristic: if the string has many single-letter fragments, join them.
    """
    if not s or len(s) > 40:
        return s
    tokens = s.split()
    # If most tokens are single letters, join everything
    single = sum(1 for t in tokens if len(t) == 1)
    if tokens and single / len(tokens) > 0.4:
        return "".join(tokens)
    return s


# Apply cleaning
df_clean = df.copy()

# 1. Clean every string column
for col in df_clean.columns:
    if col == "source_page":
        continue
    df_clean[col] = df_clean[col].apply(clean_cell)

# 2. Fix garbled ward strings
df_clean["ward"] = df_clean["ward"].apply(fix_interleaved_ward)

# 3. Drop exact duplicates
before = len(df_clean)
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
print(f"Dropped {before - len(df_clean)} duplicate row(s)")

# 4. Convert source_page to int
df_clean["source_page"] = df_clean["source_page"].astype(int)

print(f"\nCleaned shape: {df_clean.shape}")
print(f"\nSample (first 5):")
display(df_clean.head(5))

Dropped 1 duplicate row(s)

Cleaned shape: (190, 12)

Sample (first 5):


,section_num,chiefdom,constituency,sn,project_name,village,ward,chiefdom_in_row,resources,priority,sub_item,source_page
0,3.3.2.1,Kalindawalo,Petauke Central Constituency,2,Construction/Rehabilitation of Dams (Irrigatio...,Selected villages,All wards,Kalindawalo,,2,Streams,75
1,3.3.2.1,Kalindawalo,Petauke Central Constituency,2,Construction/Rehabilitation of Dams (Irrigatio...,Selected villages,All wards,Kalindawalo,,2,Unskilled Labour,75
2,3.3.2.1,Kalindawalo,Petauke Central Constituency,2,Construction/Rehabilitation of Dams (Irrigatio...,Selected villages,All wards,Kalindawalo,,2,Crushed stones,75
3,3.3.2.1,Kalindawalo,Petauke Central Constituency,2,Construction/Rehabilitation of Dams (Irrigatio...,Selected villages,All wards,Kalindawalo,,2,River/Building Sand,75
4,3.3.2.1,Kalindawalo,Petauke Central Constituency,2,Construction/Rehabilitation of Dams (Irrigatio...,Selected villages,All wards,Kalindawalo,,2,defunct dams,75


### 8.3 Save cleaned IDP projects

The cleaned DataFrame is saved to `data/cleaned/` with the same filename
for traceability.

In [50]:
cleaned_file = cleaned / "db-unza26-csc4792-petauke_lusangazi_idp_projects.csv"
df_clean.to_csv(cleaned_file, sep="|", index=False, encoding="utf-8")

print(f"Saved: {cleaned_file.name}")
print(f"Rows : {len(df_clean)}")
print(f"Path : {cleaned_file.resolve()}")

Saved: db-unza26-csc4792-petauke_lusangazi_idp_projects.csv
Rows : 190
Path : C:\Users\david\project_team_31\data\cleaned\db-unza26-csc4792-petauke_lusangazi_idp_projects.csv


### 8.4 Cleaning — IDP Wards

The ward data was already well-structured because we used a regex-based
parser during extraction. Cleaning steps:

- Strip whitespace (safety check).
- Convert `n1` and `n2` from strings to integers.
- Keep `facility` empty where no facility was recorded.

In [52]:
wards_clean = wards_df.copy()

# 1. Strip whitespace across all string columns
for col in wards_clean.columns:
    if col in ("n1", "n2", "source_page"):
        continue
    wards_clean[col] = wards_clean[col].astype(str).str.strip()

# 2. Convert n1, n2 to integers
wards_clean["n1"] = pd.to_numeric(wards_clean["n1"], errors="coerce").astype("Int64")
wards_clean["n2"] = pd.to_numeric(wards_clean["n2"], errors="coerce").astype("Int64")

print(f"Cleaned shape: {wards_clean.shape}")
print(f"Data types:")
print(wards_clean.dtypes)
print(f"\nSample:")
display(wards_clean.head(10))

Cleaned shape: (20, 8)
Data types:
sn                str
ward              str
constituency      str
district          str
n1              Int64
n2              Int64
facility          str
source_page       str
dtype: object

Sample:


,sn,ward,constituency,district,n1,n2,facility,source_page
0,1,Kaumbwe,Kaumbwe,Petauke,1,3,Rural Health Centers 1 hospital,27
1,2,Ongolwe,Petauke Central,Petauke,2,4,,27
2,3,Lusinde,Kaumbwe,Petauke,1,3,,27
3,4,Mawanda,Msanzala,Lusangazi,4,8,,27
4,5,Mbala,Petauke Central,Petauke,3,4,1general hospital,27
5,6,Mateyo Mzeka,Petauke Central,Petauke,3,3,,27
6,7,Chisangu,Msanzala,Lusangazi,1,3,,27
7,8,Msumbazi,Petauke Central,Petauke,1,6,,27
8,9,Singozi,Petauke Central,Petauke,2,3,,27
9,10,Manyane,Kaumbwe,Petauke,4,4,,27


### 8.5 Save cleaned IDP wards

In [53]:
wards_file_clean = cleaned / "db-unza26-csc4792-petauke_lusangazi_idp_wards.csv"
wards_clean.to_csv(wards_file_clean, sep="|", index=False, encoding="utf-8")

print(f"Saved: {wards_file_clean.name}")
print(f"Rows : {len(wards_clean)}")

Saved: db-unza26-csc4792-petauke_lusangazi_idp_wards.csv
Rows : 20


### 8.6 Cleaning — IDP Locations

The locations dataset from pages 42–43 has **interleaved text** in the
`status` column for several rows — a pdfplumber text-layer artifact where
two PDF text streams were merged character-by-character (e.g.
`fAieblodust 1,605.75 HA` instead of `About 1,605.75 HA`).

Instead of trying to reverse-engineer the interleave, we **strip the
garbled prefix** and rebuild the status with a clean "About" preamble
followed by the measurable facts. The words after the numeric portion
(e.g. "encroached for mostly") are already clean.

Steps:
1. Strip whitespace from all string columns.
2. Clean the `status` column using the above heuristic.
3. Convert `area_ha` and `perimeter_km` to numeric.

In [55]:
import re

def clean_status(s):
    """Clean up the garbled prefix in the status column."""
    if not s:
        return ""
    s = " ".join(str(s).split())  # collapse whitespace

    # Look for a clean "NUMBER HA" pattern — everything before it is garbled
    m = re.search(r"([\d,]+(?:\.\d+)?)\s*HA\b", s, re.IGNORECASE)
    if m:
        # Everything before the match is the garbled prefix — drop it
        return "About " + s[m.start():]
    return s


loc_clean = loc_df.copy()

# 1. Strip whitespace on all string columns
for col in loc_clean.columns:
    if col == "source_page":
        continue
    loc_clean[col] = loc_clean[col].astype(str).str.strip()

# 2. Clean status
loc_clean["status"] = loc_clean["status"].apply(clean_status)

# 3. Numeric conversions
loc_clean["area_ha"] = pd.to_numeric(
    loc_clean["area_ha"].str.replace(",", "", regex=False),
    errors="coerce"
).astype("Int64")

loc_clean["perimeter_km"] = pd.to_numeric(
    loc_clean["perimeter_km"].str.replace(",", "", regex=False).str.replace("(KM)", "", regex=False).str.strip(),
    errors="coerce"
)

print("Cleaned shape:", loc_clean.shape)
print("\nData types:")
print(loc_clean.dtypes)
print("\nFull cleaned table:")
display(loc_clean)

Cleaned shape: (10, 8)

Data types:
sn                  str
location            str
name                str
category            str
area_ha           Int64
perimeter_km    float64
status              str
source_page         str
dtype: object

Full cleaned table:


,sn,location,name,category,area_ha,perimeter_km,status,source_page
0,1,Petauke,FOREST Mupya West,LF,308,7.3,About 261.8 HA has been cleared for agricultur...,42
1,2,Petauke,Kapungwe West,LF,1348,17.4,About 808.8 HA encroached for agriculture fields,42
2,3,Petauke,Luwenga,LF,1303,15.6,About 390.9 HA encroached d for agriculture,42
3,4,Petauke,Msumbazi,LF,2141,19.6,"About 1,605.75 HA encroached for mostly",42
4,5,Petauke,Nsangwa North,LF,809,18.1,aAgbroicuutl 3tu6r4e. 0fi5e lHdsA a nd a few s...,42
5,6,Petauke,Nsangwa South,LF,1959,22.2,About 7 HA encroached with,42
6,7,Petauke,Mpamadzi,LF,791,13.2,sAebttoleumt 2e7n6ts. 8a5n dH fAie lds encroac...,42
7,8,Lusangazi,Sasare,LF,2600,26.0,"About 1,300 HA encroached with fields, village...",42
8,9,Petauke,Minga,NF,6653,33.4,"About 3,659.15 HA encroached with agricultural",42
9,10,Petauke,FOREST Mvuvye East,NF,35937,132.0,"About 14,375.8 HA encroached in some parts wit...",43


### 8.7 Refining the location status cleaner

The initial clean_status() heuristic was too aggressive: it replaced
garbled prefixes even when the numeric portion was itself interleaved,
losing precision (e.g. `5,887.7` became `7`).

We make it conservative: only rebuild the status when the numeric
portion contains at least 3 digits, indicating a clean value.

In [56]:
def clean_status_safe(s):
    """Conservative status cleaner — only rebuild if number is trustworthy."""
    if not s:
        return ""
    s = " ".join(str(s).split())

    # Look for a clean numeric pattern with at least 3 digits total
    m = re.search(r"(\d[\d,]{2,}(?:\.\d+)?)\s*HA\b", s, re.IGNORECASE)
    if m:
        return "About " + s[m.start():]
    # Leave as-is if we can't confidently find the numeric portion
    return s


# Re-do from the raw extracted data
loc_clean = loc_df.copy()

for col in loc_clean.columns:
    if col == "source_page":
        continue
    loc_clean[col] = loc_clean[col].astype(str).str.strip()

loc_clean["status"] = loc_clean["status"].apply(clean_status_safe)

loc_clean["area_ha"] = pd.to_numeric(
    loc_clean["area_ha"].str.replace(",", "", regex=False),
    errors="coerce"
).astype("Int64")

loc_clean["perimeter_km"] = pd.to_numeric(
    loc_clean["perimeter_km"]
        .str.replace("(KM)", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip(),
    errors="coerce"
)

print("Shape:", loc_clean.shape)
display(loc_clean[["sn", "name", "status"]])

Shape: (10, 8)


,sn,name,status
0,1,FOREST Mupya West,About 261.8 HA has been cleared for agricultur...
1,2,Kapungwe West,About 808.8 HA encroached for agriculture fields
2,3,Luwenga,About 390.9 HA encroached d for agriculture
3,4,Msumbazi,"About 1,605.75 HA encroached for mostly"
4,5,Nsangwa North,aAgbroicuutl 3tu6r4e. 0fi5e lHdsA a nd a few s...
5,6,Nsangwa South,sAebttoleumt 5e8n7t .7 HA encroached with
6,7,Mpamadzi,sAebttoleumt 2e7n6ts. 8a5n dH fAie lds encroac...
7,8,Sasare,"About 1,300 HA encroached with fields, village..."
8,9,Minga,"About 3,659.15 HA encroached with agricultural"
9,10,FOREST Mvuvye East,"About 14,375.8 HA encroached in some parts wit..."


### 8.8 Save cleaned IDP locations

Saving the cleaned dataset. Known limitation: 3 rows still contain
interleaved characters in the `status` column where pdfplumber merged two
text streams beyond recovery. These rows preserve the row identity and the
other columns; the affected status strings are marked in the notebook's
Known Issues table.

In [57]:
loc_file_clean = cleaned / "db-unza26-csc4792-petauke_lusangazi_idp_locations.csv"
loc_clean.to_csv(loc_file_clean, sep="|", index=False, encoding="utf-8")

print(f"Saved: {loc_file_clean.name}")
print(f"Rows : {len(loc_clean)}")

Saved: db-unza26-csc4792-petauke_lusangazi_idp_locations.csv
Rows : 10


### 8.9 Cleaning — IDP Objectives

The objectives dataset was extracted from page 71 of the Joint IDP.
Inspection issues:

- Some `strategy` values retain leading bullet characters (`•` or
  `\uf0b7`) from the source PDF.
- Whitespace in `objective` and `strategy` columns.

In [58]:
obj_file = processed / "db-unza26-csc4792-petauke_lusangazi_idp_objectives.csv"
obj_df = pd.read_csv(obj_file, sep="|", dtype=str, keep_default_na=False)

print("Shape:", obj_df.shape)
print("Columns:", list(obj_df.columns))
print("\nEmpty values per column:")
print((obj_df == "").sum())
print("\nDuplicates:", obj_df.duplicated().sum())
print("\nFull table:")
display(obj_df)

Shape: (9, 4)
Columns: ['sn', 'objective', 'strategy', 'source_page']

Empty values per column:
sn             0
objective      0
strategy       0
source_page    0
dtype: int64

Duplicates: 0

Full table:


,sn,objective,strategy,source_page
0,1,"To increase and diversify agriculture, livesto...","• Diversification of agricultural, livestock a...",71
1,1,"To increase and diversify agriculture, livesto...",Creation of crop demo plots and crop field sch...,71
2,1,"To increase and diversify agriculture, livesto...",Introduce intensive and commercial farming usi...,71
3,1,"To increase and diversify agriculture, livesto...",Introduce sustainable and environmentally soun...,71
4,1,"To increase and diversify agriculture, livesto...",Strengthening emergency preparedness through t...,71
5,1,"To increase and diversify agriculture, livesto...",Conduct continuous sensitization meetings to f...,71
6,2,To promote the conservation of natural resources,• Creation of eco-friendly forests/fruit plant...,71
7,3,To develop infrastructure in the key sectors o..., Develop infrastructure in the key sectors of...,71
8,3,To develop infrastructure in the key sectors o...,Promotion of public private partnerships in th...,71


### 8.10 Cleaning — IDP Objectives

Issues identified:

- Three `strategy` values retain a leading bullet character (`•` or the
  private-use character `\uf0b7`) from the source PDF's bullet lists.

Steps:
- Strip leading bullets from `strategy`.
- Strip whitespace.
- Convert `source_page` to integer.

In [59]:
obj_clean = obj_df.copy()

# 1. Strip leading bullets and whitespace from strategy
obj_clean["strategy"] = (
    obj_clean["strategy"]
        .astype(str)
        .str.replace(r"^[\u2022\uf0b7•]+\s*", "", regex=True)
        .str.strip()
)

# 2. Strip whitespace on remaining string columns
for col in ["objective"]:
    obj_clean[col] = obj_clean[col].astype(str).str.strip()

# 3. Numeric conversions
obj_clean["sn"] = pd.to_numeric(obj_clean["sn"], errors="coerce").astype("Int64")
obj_clean["source_page"] = pd.to_numeric(obj_clean["source_page"], errors="coerce").astype("Int64")

print("Shape:", obj_clean.shape)
print("\nData types:")
print(obj_clean.dtypes)
print("\nCleaned table:")
display(obj_clean)

Shape: (9, 4)

Data types:
sn             Int64
objective        str
strategy         str
source_page    Int64
dtype: object

Cleaned table:


,sn,objective,strategy,source_page
0,1,"To increase and diversify agriculture, livesto...","Diversification of agricultural, livestock and...",71
1,1,"To increase and diversify agriculture, livesto...",Creation of crop demo plots and crop field sch...,71
2,1,"To increase and diversify agriculture, livesto...",Introduce intensive and commercial farming usi...,71
3,1,"To increase and diversify agriculture, livesto...",Introduce sustainable and environmentally soun...,71
4,1,"To increase and diversify agriculture, livesto...",Strengthening emergency preparedness through t...,71
5,1,"To increase and diversify agriculture, livesto...",Conduct continuous sensitization meetings to f...,71
6,2,To promote the conservation of natural resources,Creation of eco-friendly forests/fruit plantat...,71
7,3,To develop infrastructure in the key sectors o...,Develop infrastructure in the key sectors of t...,71
8,3,To develop infrastructure in the key sectors o...,Promotion of public private partnerships in th...,71


### 8.11 Save cleaned IDP objectives

In [60]:
obj_file_clean = cleaned / "db-unza26-csc4792-petauke_lusangazi_idp_objectives.csv"
obj_clean.to_csv(obj_file_clean, sep="|", index=False, encoding="utf-8")

print(f"Saved: {obj_file_clean.name}")
print(f"Rows : {len(obj_clean)}")

Saved: db-unza26-csc4792-petauke_lusangazi_idp_objectives.csv
Rows : 9


### 8.12 Cleaning — Strategic Plan: Population

The population dataset was extracted from page 11 of the Strategic Plan.
Expected cleaning:

- `year` and `urban_pct` should be numeric, not strings.
- Strip whitespace.

In [61]:
pop_file = processed / "db-unza26-csc4792-petauke_stratplan_population.csv"
pop_df = pd.read_csv(pop_file, sep="|", dtype=str, keep_default_na=False)

print("Shape:", pop_df.shape)
print("Columns:", list(pop_df.columns))
print("\nEmpty values per column:")
print((pop_df == "").sum())
print("\nDuplicates:", pop_df.duplicated().sum())
print("\nFull table:")
display(pop_df)

Shape: (30, 4)
Columns: ['province', 'year', 'urban_pct', 'source_page']

Empty values per column:
province       0
year           0
urban_pct      0
source_page    0
dtype: int64

Duplicates: 0

Full table:


,province,year,urban_pct,source_page
0,Central,2011,25.5,11
1,Central,2015,25.4,11
2,Central,2020,25.4,11
3,Copperbelt,2011,82.1,11
4,Copperbelt,2015,83.0,11
5,Copperbelt,2020,84.1,11
6,Luapula,2011,19.4,11
7,Luapula,2015,21.0,11
8,Luapula,2020,23.1,11
9,Lusaka,2011,85.3,11


### 8.12 Cleaning — Strategic Plan: Population

Issues identified:

- `year` and `urban_pct` are stored as strings — convert to integers
  and floats respectively.
- Province name "North- Western" has a stray space — normalise to
  "North-Western".
- Whitespace on province names.

In [62]:
pop_clean = pop_df.copy()

# 1. Strip whitespace on province
pop_clean["province"] = pop_clean["province"].astype(str).str.strip()

# 2. Fix "North- Western" -> "North-Western"
pop_clean["province"] = pop_clean["province"].str.replace(
    r"\s*-\s*", "-", regex=True
)

# 3. Numeric conversions
pop_clean["year"] = pd.to_numeric(pop_clean["year"], errors="coerce").astype("Int64")
pop_clean["urban_pct"] = pd.to_numeric(pop_clean["urban_pct"], errors="coerce").astype(float)
pop_clean["source_page"] = pd.to_numeric(pop_clean["source_page"], errors="coerce").astype("Int64")

print("Shape:", pop_clean.shape)
print("\nData types:")
print(pop_clean.dtypes)
print("\nUnique provinces:")
print(sorted(pop_clean["province"].unique()))
print("\nCleaned table (head):")
display(pop_clean.head(10))

Shape: (30, 4)

Data types:
province           str
year             Int64
urban_pct      float64
source_page      Int64
dtype: object

Unique provinces:
['Central', 'Copperbelt', 'Luapula', 'Lusaka', 'Muchinga', 'North-Western', 'Northern', 'Southern', 'Western', 'Zambia']

Cleaned table (head):


,province,year,urban_pct,source_page
0,Central,2011,25.5,11
1,Central,2015,25.4,11
2,Central,2020,25.4,11
3,Copperbelt,2011,82.1,11
4,Copperbelt,2015,83.0,11
5,Copperbelt,2020,84.1,11
6,Luapula,2011,19.4,11
7,Luapula,2015,21.0,11
8,Luapula,2020,23.1,11
9,Lusaka,2011,85.3,11


### 8.13 Save cleaned population dataset

In [63]:
pop_file_clean = cleaned / "db-unza26-csc4792-petauke_stratplan_population.csv"
pop_clean.to_csv(pop_file_clean, sep="|", index=False, encoding="utf-8")

print(f"Saved: {pop_file_clean.name}")
print(f"Rows : {len(pop_clean)}")

Saved: db-unza26-csc4792-petauke_stratplan_population.csv
Rows : 30


### 8.14 Cleaning — Strategic Plan: Employment

Issues identified:

- Numeric columns contain thousand separators (`3,398,294`) — remove
  commas and convert to integers.
- `potential_labour_force` has 3 empty cells (only recorded for the
  Local Perspective) — leave as null.
- Strip whitespace on `perspective` and `category`.

In [65]:
emp_clean = emp_df.copy()

# 1. Strip whitespace on string columns
for col in ["perspective", "category"]:
    emp_clean[col] = emp_clean[col].astype(str).str.strip()

# 2. Numeric conversions
numeric_cols = [
    "labour_force",
    "persons_in_employment",
    "persons_in_unemployment",
    "potential_labour_force",
]

for col in numeric_cols:
    emp_clean[col] = pd.to_numeric(
        emp_clean[col].astype(str).str.replace(",", "", regex=False).str.strip(),
        errors="coerce"
    ).astype("Int64")

emp_clean["source_page"] = pd.to_numeric(emp_clean["source_page"], errors="coerce").astype("Int64")

print("Shape:", emp_clean.shape)
print("\nData types:")
print(emp_clean.dtypes)
print("\nCleaned table:")
display(emp_clean)

Shape: (6, 7)

Data types:
perspective                  str
category                     str
labour_force               Int64
persons_in_employment      Int64
persons_in_unemployment    Int64
potential_labour_force     Int64
source_page                Int64
dtype: object

Cleaned table:


,perspective,category,labour_force,persons_in_employment,persons_in_unemployment,potential_labour_force,source_page
0,International Perspective,Total,3398294,2971169,427125,<NA>,13
1,International Perspective,Male,2041306,1797957,243349,<NA>,13
2,International Perspective,Female,1356988,1173212,183776,<NA>,13
3,Local Perspective,Total,5049059,2971169,427125,1650764,13
4,Local Perspective,Male,2759098,1797957,243349,717792,13
5,Local Perspective,Female,2289961,1173212,183776,932972,13


### 8.15 Save cleaned employment dataset

In [66]:
emp_file_clean = cleaned / "db-unza26-csc4792-petauke_stratplan_employment.csv"
emp_clean.to_csv(emp_file_clean, sep="|", index=False, encoding="utf-8")

print(f"Saved: {emp_file_clean.name}")
print(f"Rows : {len(emp_clean)}")

Saved: db-unza26-csc4792-petauke_stratplan_employment.csv
Rows : 6


### 8.16 Cleaning — Strategic Plan: Health

Issues identified:

- `cases` values contain thousands separators with spaces (e.g. `51, 309`)
  — remove spaces and commas, convert to integers.
- Disease name `Malaria conﬁrmed cases` uses a ligature character `ﬁ` (a
  single Unicode glyph representing `fi`) — replace with `fi`.
- Strip whitespace on `disease`.

In [68]:
health_clean = health_df.copy()

# 1. Strip whitespace on disease
health_clean["disease"] = health_clean["disease"].astype(str).str.strip()

# 2. Fix ligature characters (ﬁ, ﬂ, ﬀ, etc.)
ligature_map = {
    "\ufb01": "fi",   # ﬁ
    "\ufb02": "fl",   # ﬂ
    "\ufb00": "ff",   # ﬀ
    "\ufb03": "ffi",  # ﬃ
    "\ufb04": "ffl",  # ﬄ
}
for lig, repl in ligature_map.items():
    health_clean["disease"] = health_clean["disease"].str.replace(lig, repl, regex=False)

# 3. Numeric conversion — remove spaces and commas
health_clean["cases"] = pd.to_numeric(
    health_clean["cases"].astype(str).str.replace(" ", "", regex=False).str.replace(",", "", regex=False),
    errors="coerce"
).astype("Int64")

health_clean["incidence_rate"] = pd.to_numeric(
    health_clean["incidence_rate"], errors="coerce"
).astype("Int64")

health_clean["sn"] = pd.to_numeric(health_clean["sn"], errors="coerce").astype("Int64")
health_clean["source_page"] = pd.to_numeric(health_clean["source_page"], errors="coerce").astype("Int64")

print("Shape:", health_clean.shape)
print("\nData types:")
print(health_clean.dtypes)
print("\nCleaned table:")
display(health_clean)

Shape: (10, 5)

Data types:
sn                Int64
disease             str
cases             Int64
incidence_rate    Int64
source_page       Int64
dtype: object

Cleaned table:


,sn,disease,cases,incidence_rate,source_page
0,1,Respiratory infections,51309,181,15
1,2,Malaria conrmed cases,41419,173,15
2,3,Muscular skeleton & connective tissues,10745,38,15
3,4,Digestive systems,7046,25,15
4,5,Diarrhea non blood,6152,22,15
5,6,Pyrexia of unknown origin (PUO),3895,14,15
6,7,"Trauma, other injuries & wounds",2547,9,15
7,8,Skin diseases not infectious,2473,9,15
8,9,dental carries,1666,6,15
9,10,Throat diseases,1509,5,15


### 8.17 Fixing the private-use ligature U+F001

The corrupted character in "Malaria con?rmed" is `U+F001` (private-use
area), used by the PDF's embedded font to represent the `fi` ligature.
We replace it with the two-character sequence `fi`.

In [70]:
# Replace U+F001 with "fi" in the disease column
health_clean["disease"] = health_clean["disease"].str.replace(
    "\uf001", "fi", regex=False
)

# Verify
print("Fixed row 1:", health_clean.loc[1, "disease"])

# Also scan the rest of the dataframe for any other U+F001 or similar
for col in health_clean.select_dtypes(include="object").columns:
    mask = health_clean[col].str.contains("\uf001", na=False)
    if mask.any():
        print(f"\nU+F001 also found in column: {col}")
        print(health_clean.loc[mask, col])

Fixed row 1: Malaria confirmed cases


C:\Users\david\AppData\Local\Temp\ipykernel_4160\4232632156.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in health_clean.select_dtypes(include="object").columns:


### 8.18 Save cleaned health dataset

In [71]:
health_file_clean = cleaned / "db-unza26-csc4792-petauke_stratplan_health.csv"
health_clean.to_csv(health_file_clean, sep="|", index=False, encoding="utf-8")

print(f"Saved: {health_file_clean.name}")
print(f"Rows : {len(health_clean)}")

Saved: db-unza26-csc4792-petauke_stratplan_health.csv
Rows : 10


### 8.19 Summary — Data Cleaning Complete

All 7 datasets extracted from the IDP and Strategic Plan PDFs have been
cleaned and saved to `data/cleaned/`.

In [72]:
from pathlib import Path

print(f"{'File':65s} {'Rows':>6s}")
print("-" * 75)

for f in sorted(cleaned.glob("db-unza26-csc4792-*.csv")):
    df = pd.read_csv(f, sep="|", dtype=str, keep_default_na=False)
    print(f"{f.name:65s} {len(df):>6d}")

File                                                                Rows
---------------------------------------------------------------------------
db-unza26-csc4792-petauke_lusangazi_idp_locations.csv                 10
db-unza26-csc4792-petauke_lusangazi_idp_objectives.csv                 9
db-unza26-csc4792-petauke_lusangazi_idp_projects.csv                 190
db-unza26-csc4792-petauke_lusangazi_idp_wards.csv                     20
db-unza26-csc4792-petauke_stratplan_employment.csv                     6
db-unza26-csc4792-petauke_stratplan_health.csv                        10
db-unza26-csc4792-petauke_stratplan_population.csv                    30
